<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<span style="display:inline-block;margin-bottom:10px;padding:2px 12px;border-radius:999px;background:#6366f1;color:#fff;font:600 11px/20px system-ui,sans-serif;letter-spacing:.05em;text-transform:uppercase">Step 9</span>
<h1 style="font-size:1.35em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Step 9: The server</h1>
<p style="margin:.55em 0">One process relays each browser conversation to OpenAI, so the key never leaves it. Production concerns live here:
logs tagged with the conversation id (text or JSON), a cap on concurrent calls, a time limit per call, an origin
allowlist, frame size limits, <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">/healthz</code> and <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">/metrics</code>, and a graceful drain on shutdown.</p>

</div>

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Logging</h2><p style="margin:.55em 0">A contextvar carries the conversation id into every log line, including from worker threads.</p>

</div>

In [1]:
%%writefile build/app/logs.py
"""Logging: plain text for development, one JSON object per line for production log pipelines.
Every line carries the conversation id (a contextvar set per connection; tasks and worker threads inherit it)."""
import contextvars
import json
import logging
import sys
import time

session_id: contextvars.ContextVar[str] = contextvars.ContextVar("session_id", default="-")


class _SessionId(logging.Filter):
    def filter(self, record):
        record.session = session_id.get()
        return True


class _Json(logging.Formatter):
    def format(self, r):
        out = {"ts": time.strftime("%Y-%m-%dT%H:%M:%S", time.gmtime(r.created)) + f".{int(r.msecs):03d}Z",
               "level": r.levelname, "logger": r.name, "session": getattr(r, "session", "-"), "msg": r.getMessage()}
        if r.exc_info:
            out["exc"] = self.formatException(r.exc_info)
        return json.dumps(out, ensure_ascii=False)


def setup(level: str = "INFO", fmt: str = "text") -> None:
    handler = logging.StreamHandler(sys.stderr)
    handler.addFilter(_SessionId())
    handler.setFormatter(_Json() if fmt == "json" else
                         logging.Formatter("%(asctime)s %(levelname)s [%(session)s] %(name)s: %(message)s"))
    root = logging.getLogger()
    root.handlers[:] = [handler]
    root.setLevel(level.upper())
    logging.getLogger("websockets").setLevel(logging.WARNING)  # every page load is a "rejected" handshake


Writing build/app/logs.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">The browser as a speaker</h2>
<p style="margin:.55em 0">The model's audio plays in the browser, so barge-in needs to know how much the browser actually played. The page
reports it every 50 ms; between reports we extrapolate a little.</p>

</div>

In [2]:
%%writefile build/app/server.py
"""The server: one process, one websocket per browser conversation, relayed to OpenAI Realtime.

  GET /            the page (app/static/index.html)
  GET /healthz     liveness/readiness: 200 {"status": "ok", ...}, 503 while draining
  GET /metrics     Prometheus text
  WS  /ws?agent=   a conversation with one configured agent

Browser mic (PCM16 24 kHz binary frames) -> here -> OpenAI; model audio back to the browser.
The API key stays in this process. Limits: concurrent conversations, time per conversation, frame size,
browser origins. On SIGTERM/Ctrl-C: stop accepting, let running conversations finish for drain_s, then exit.
"""
import asyncio
import contextlib
import http
import json
import logging
import signal
import time
from pathlib import Path
from urllib.parse import parse_qs, urlsplit

from websockets.asyncio.server import serve
from websockets.exceptions import ConnectionClosed

from app import cascade, providers, realtime, registry, store, webrtc
from app.logs import session_id
from app.metrics import METRICS
from app.session import Session
from app.settings import Settings, api_key

log = logging.getLogger(__name__)
INDEX = Path(__file__).with_name("static") / "index.html"
EXTRAPOLATE_S = 0.2  # max time we assume playback ran on since the browser's last report


class BrowserSpeaker:
    """The `speaker` converse() expects, but the sound plays in the browser.

    All model audio forms one byte stream. `position` = where the next write lands; `played` = what the browser
    reports it has played (~every 50 ms), extrapolated in real time for at most EXTRAPOLATE_S."""

    def __init__(self, out: asyncio.Queue):
        self.out = out          # messages for the browser (bytes = audio, dict = JSON)
        self.written = 0
        self.dropped = 0
        self.reported, self.reported_at = 0, time.monotonic()

    def report(self, played: int) -> None:
        self.reported, self.reported_at = played, time.monotonic()

    def correct(self, estimate: int, actual: int) -> None:
        """The browser says how much a clear() really dropped; fix our estimate."""
        self.dropped += actual - estimate

    @property
    def position(self) -> int:
        return self.written - self.dropped

    @property
    def played(self) -> int:
        ahead = min(time.monotonic() - self.reported_at, EXTRAPOLATE_S)
        return min(self.reported + int(ahead * realtime.BYTES_PER_S) // 2 * 2, self.position)

    def write(self, b: bytes) -> None:
        self.written += len(b)
        self.out.put_nowait(b)

    def clear(self) -> int:
        n = max(self.position - self.played, 0)
        self.dropped += n
        self.out.put_nowait({"type": "clear", "estimate": n})
        return n

    def buffered_seconds(self) -> float:
        return (self.position - self.played) / realtime.BYTES_PER_S


class BrowserGone(Exception):
    pass


def _count(v) -> bool:
    return type(v) is int and v >= 0  # exact int: True/False are ints too


Writing build/app/server.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">The app</h2>
<p style="margin:.55em 0"><code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">handler</code> runs one conversation: agent allowlist, capacity, a Session, the resilient conversation, and one
plain sentence to the browser for any failure (the traceback goes to the log). <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">http</code> serves the page, health and
metrics. <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">drain</code> stops taking calls and lets running ones finish.</p>

</div>

In [3]:
%%writefile -a build/app/server.py

class App:
    """Process-wide state: settings, the key, the filler clip, and how many conversations are running."""

    def __init__(self, settings: Settings, key: str | None = None, start_session=None, parts=None, rtc=None):
        self.settings, self.key = settings, key
        self.active = 0
        self.draining = False
        self.done = asyncio.Event()
        filler = settings.skills.filler_audio and settings.path(settings.skills.filler_audio)
        self.filler = filler.read_bytes() if filler and filler.is_file() else None
        if filler and not self.filler:
            log.warning("no filler audio at %s: tool calls will be silent until the answer", filler)
        # injectable for tests (a fake OpenAI); production opens real Realtime sessions
        self.start_session = start_session or (lambda instr, tools, vad: realtime.start_session(
            self.settings, self.key, instr, tools, vad))
        # the cascade engine's providers (app/providers/), checked at startup; injectable for tests
        self.parts = parts or (providers.load(settings) if settings.engine.kind == "cascade" else None)
        self.rtc = rtc or {}  # webrtc engine test hooks: create / connect / hangup

    async def handler(self, browser):
        session_id.set(f"{id(browser) & 0xffffff:06x}")
        s = self.settings
        out: asyncio.Queue = asyncio.Queue()
        agent = parse_qs(urlsplit(browser.request.path).query).get("agent", [""])[0]
        if agent not in s.agents:  # allowlist
            await browser.send(json.dumps({"type": "error", "message": f"unknown agent '{agent}'"}))
            return
        if self.draining or self.active >= s.server.max_sessions:
            METRICS.inc("sessions_rejected_total")
            await browser.send(json.dumps({"type": "error", "reconnect": True,
                                           "message": "The desk is busy right now. Please try again in a minute."}))
            return
        self.active += 1
        METRICS.inc("sessions_total", agent=agent)
        METRICS.set("sessions_active", self.active)
        loop = asyncio.get_running_loop()
        # skills run on worker threads: hand their messages to the loop thread-safely
        session = Session(s, agent, emit=lambda m: loop.call_soon_threadsafe(out.put_nowait, m))
        spk = BrowserSpeaker(out)
        started, opened = time.monotonic(), []
        offers: asyncio.Queue = asyncio.Queue(maxsize=1)  # webrtc engine: the page's SDP offer

        async def to_browser():
            while (m := await out.get()) is not None:
                await browser.send(m if isinstance(m, bytes) else json.dumps(m))

        async def frames():
            async for m in browser:
                if isinstance(m, bytes):
                    if m and len(m) % 2 == 0:  # whole PCM16 samples only
                        yield m
                    continue
                try:
                    c = json.loads(m)
                except ValueError:
                    c = None
                if isinstance(c, dict) and set(c) == {"played"} and _count(c["played"]):
                    spk.report(c["played"])
                elif isinstance(c, dict) and set(c) == {"cleared", "estimate"} and _count(c["cleared"]) \
                        and _count(c["estimate"]):
                    spk.correct(c["estimate"], c["cleared"])
                elif isinstance(c, dict) and set(c) == {"sdp"} and isinstance(c["sdp"], str) \
                        and 0 < len(c["sdp"]) <= webrtc.MAX_SDP and offers.empty():
                    offers.put_nowait(c["sdp"])
                else:
                    log.warning("ignored browser message %.80r", m)

        async def mic():
            try:
                async for m in frames():
                    yield m
            except ConnectionClosed:  # tab closed, laptop asleep: not a server failure
                pass
            raise BrowserGone  # ends the conversation at once instead of at the time limit

        def ready():
            log.info("%s conversation started (%s engine)", agent, s.engine.kind)
            out.put_nowait({"type": "status", "state": "ready", "limit_s": s.server.session_limit_s, "agent": agent})

        async def start():  # called again by converse_resilient after an OpenAI stall
            ws = await self.start_session(session.instructions(), session.specs(), session.config.vad)
            if not opened:
                ready()
            opened.append(ws)
            return ws

        def on_note(t, event, detail):
            out.put_nowait({"type": "log", "t": t, "event": event, "detail": str(detail)})

        sender = asyncio.create_task(to_browser())
        try:
            out.put_nowait({"type": "status", "state": "connecting", "engine": s.engine.kind})
            filler = self.filler if session.config.filler else None
            if s.engine.kind == "webrtc":
                await webrtc.converse(
                    s, self.key, offers, lambda sdp: out.put(({"type": "rtc_answer", "sdp": sdp})), mic(),
                    seconds=s.server.session_limit_s, instructions=session.instructions(), tools=session.specs(),
                    run_tool=session.run_tool, vad=session.config.vad, on_note=on_note, on_ready=ready,
                    max_resumes=s.resilience.max_resumes, **self.rtc)
            elif s.engine.kind == "cascade":
                await cascade.converse(
                    self.parts, s, self.key, mic(), spk, seconds=s.server.session_limit_s,
                    instructions=session.instructions(), tools=session.specs(), run_tool=session.run_tool,
                    vad=session.config.vad, filler_pcm=filler, on_note=on_note, on_ready=ready,
                    max_resumes=s.resilience.max_resumes)
            else:
                await realtime.converse_resilient(
                    start, mic(), spk, seconds=s.server.session_limit_s, max_resumes=s.resilience.max_resumes,
                    context=session.context, on_note=on_note, run_tool=session.run_tool, filler_pcm=filler,
                    stall_s=s.resilience.stall_s, ping_timeout_s=s.resilience.ping_timeout_s)
            out.put_nowait({"type": "ended", "reason": f"Time limit of {s.server.session_limit_s // 60} min reached."})
        except BrowserGone:
            log.info("browser left after %.0f s", time.monotonic() - started)
        except realtime.DROPPED as e:  # only the OpenAI link gets here, and only after the reconnects ran out
            METRICS.inc("session_errors_total", kind="openai_dropped")
            log.warning("OpenAI connection lost after %.0f s, %d connection(s): %r",
                        time.monotonic() - started, len(opened), e)
            out.put_nowait({"type": "error", "reconnect": True,
                            "message": "The connection to OpenAI kept dropping (network or OpenAI side). "
                                       "Reconnect to start a new conversation."})
        except Exception:  # one catch-all: full traceback in the log, one plain sentence to the browser
            METRICS.inc("session_errors_total", kind="internal")
            log.exception("conversation failed")
            out.put_nowait({"type": "error", "reconnect": True,
                            "message": "Something went wrong on the server (details in its log). "
                                       "Reconnect to start a new conversation."})
        finally:
            self.active -= 1
            METRICS.set("sessions_active", self.active)
            if self.draining and not self.active:
                self.done.set()
            out.put_nowait(None)
            try:
                await asyncio.wait_for(sender, 2)  # flush the last messages
            except Exception:
                sender.cancel()

    def http(self, connection, request):
        """Plain HTTP routes; /ws goes on to the websocket handshake."""
        path = urlsplit(request.path).path
        if path == "/ws":
            return None
        if path == "/healthz":
            status = http.HTTPStatus.SERVICE_UNAVAILABLE if self.draining else http.HTTPStatus.OK
            return self._respond(connection, status, json.dumps(
                {"status": "draining" if self.draining else "ok", "active": self.active,
                 "agents": sorted(self.settings.agents)}), "application/json")
        if path == "/metrics":
            return self._respond(connection, http.HTTPStatus.OK, METRICS.render(), "text/plain; version=0.0.4")
        if path in ("/", "/index.html"):
            return self._respond(connection, http.HTTPStatus.OK, INDEX.read_text(encoding="utf-8"),
                                 "text/html; charset=utf-8")
        return connection.respond(http.HTTPStatus.NOT_FOUND, "not found\n")

    @staticmethod
    def _respond(connection, status, body: str, content_type: str):
        r = connection.respond(status, body)
        del r.headers["Content-Type"]
        r.headers["Content-Type"] = content_type
        r.headers["Cache-Control"] = "no-store"
        return r

    async def drain(self, server) -> None:
        """Stop taking new conversations; give running ones drain_s to finish; then close."""
        self.draining = True
        log.info("draining: %d conversation(s) running", self.active)
        if self.active:
            with contextlib.suppress(asyncio.TimeoutError):
                await asyncio.wait_for(self.done.wait(), self.settings.server.drain_s)
        server.close()


Appending to build/app/server.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0"><code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">run</code> validates everything at startup (skills, prompts), opens the database, wires signals, and serves.</p>

</div>

In [4]:
%%writefile -a build/app/server.py

async def run(settings: Settings, ready: asyncio.Event | None = None, stop: asyncio.Event | None = None,
              app: App | None = None) -> None:
    """Serve until SIGTERM/SIGINT (or `stop` is set), then drain. `app` is injectable for tests."""
    tools = registry.discover()
    for name in settings.agents:  # fail at startup, not in the first call: toolsets + prompt placeholders
        Session(settings, name).instructions()
    app = app or App(settings, api_key(settings))
    store.init(settings.path(settings.store.path), seed_demo=settings.store.seed_demo)
    stop = stop or asyncio.Event()
    loop = asyncio.get_running_loop()
    for sig in (signal.SIGINT, signal.SIGTERM):
        with contextlib.suppress(NotImplementedError, ValueError):  # Windows: no loop signal handlers
            loop.add_signal_handler(sig, stop.set)
    origins = list(settings.server.origins) + [None]  # None: non-browser clients (health checks, tests)
    async with serve(app.handler, settings.server.host, settings.server.port, process_request=app.http,
                     origins=origins, max_size=settings.server.max_message_bytes) as server:
        log.info("serving http://%s:%d  engine=%s  agents=%s  toolsets=%s", settings.server.host, settings.server.port,
                 settings.engine.kind,
                 sorted(settings.agents), {k: len(v) for k, v in tools.items()})
        if ready:
            ready.set()
        try:
            await stop.wait()
        finally:
            await app.drain(server)


Appending to build/app/server.py


In [5]:
%%writefile build/app/__main__.py
"""python -m app [--config path]   (or APP_CONFIG=path python -m app)"""
import argparse
import asyncio
import sys

from app import logs, server, settings


def main() -> int:
    p = argparse.ArgumentParser(prog="python -m app", description="Voice front desk server")
    p.add_argument("--config", help="settings file (default: $APP_CONFIG or config/app.toml)")
    args = p.parse_args()
    try:
        s = settings.load(args.config)
        settings.api_key(s)  # fail fast without a key
    except settings.SettingsError as e:
        print(f"settings error: {e}", file=sys.stderr)
        return 2
    logs.setup(s.server.log_level, s.server.log_format)
    try:
        asyncio.run(server.run(s))
    except KeyboardInterrupt:  # Windows: Ctrl-C arrives here (no loop signal handlers)
        pass
    return 0


if __name__ == "__main__":
    sys.exit(main())


Writing build/app/__main__.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">The page and the filler clip</h2><p style="margin:.55em 0">The browser page (Preline UI, audio worklets) and the 1.35 s "Let me check." clip are copied as files.</p>

</div>

In [6]:
import shutil
shutil.copy('../app/static/index.html', 'build/app/static/index.html')
shutil.copy('../assets/filler_marin.pcm', 'build/assets/filler_marin.pcm')
[p.name for p in __import__('pathlib').Path('build').rglob('*') if p.suffix in ('.html', '.pcm')]

['index.html', 'filler_marin.pcm']

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Try it: the real server against a fake OpenAI</h2>

</div>

In [7]:
import sys
sys.path.insert(0, 'build')   # `import app` now means the app you are building

In [8]:
import asyncio, json, urllib.request
from websockets.asyncio.server import serve
from websockets.asyncio.client import connect
from app import realtime, server, settings
from app.metrics import METRICS

async def fake_openai(ws):
    await ws.send(json.dumps({'type': 'session.created'}))
    await ws.recv()
    await ws.send(json.dumps({'type': 'session.updated'}))
    call = {'type': 'function_call', 'name': 'list_doctors', 'call_id': 'c1', 'arguments': '{}'}
    heard = False
    async for raw in ws:
        m = json.loads(raw)
        if m['type'] == 'input_audio_buffer.append' and not heard:
            heard = True
            for e in ({'type': 'response.created', 'response': {}},
                      {'type': 'response.done', 'response': {'status': 'completed', 'output': [call]}}):
                await ws.send(json.dumps(e))
        elif m['type'] == 'response.create':
            for e in ({'type': 'response.output_audio_transcript.done', 'transcript': 'We have three doctors.'},
                      {'type': 'response.done', 'response': {'status': 'completed', 'output': []}}):
                await ws.send(json.dumps(e))

s = settings.load(env={'APP__SERVER__PORT': '8793', 'APP__OPENAI__URL': '"ws://127.0.0.1:8794"', 'APP__ENGINE__KIND': '"realtime"',
                       'APP__STORE__PATH': 'data/step8.db'})
ready, stop = asyncio.Event(), asyncio.Event()
upstream = await serve(fake_openai, '127.0.0.1', 8794)
srv = asyncio.create_task(server.run(s, ready=ready, stop=stop, app=server.App(s, key='test')))
await ready.wait()
# the server shares this notebook's event loop: blocking calls must go to a thread, or it can't answer
health = await asyncio.to_thread(urllib.request.urlopen, 'http://127.0.0.1:8793/healthz')
json.loads(health.read())

{'status': 'ok', 'active': 0, 'agents': ['front_desk', 'support']}

In [9]:
msgs = []
async with connect('ws://127.0.0.1:8793/ws?agent=front_desk') as ws:
    while json.loads(await ws.recv()).get('state') != 'ready':
        pass
    await ws.send(bytes(4800))
    while True:
        m = await asyncio.wait_for(ws.recv(), 10)
        if isinstance(m, bytes):
            continue
        msgs.append(json.loads(m))
        if msgs[-1].get('event') == 'model_said':
            break
[(m['type'], m.get('skill') or m.get('event')) for m in msgs]

[('log', 'response.created'),
 ('log', 'response.done'),
 ('log', 'tool_call'),
 ('desk', 'list_doctors'),
 ('log', 'model_said')]

In [10]:
[l for l in METRICS.render().splitlines() if 'list_doctors' in l]

['tool_calls_total{skill="list_doctors",status="ok"} 1']

In [11]:
stop.set()
await srv
upstream.close()
'stopped'

'stopped'

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0"><b>Next:</b> <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">10_production.ipynb</code>: production config, the container, and the full self-check.</p>

</div>